# The full configuration matrix

Every radius rule against every model Hessian against every subproblem solver, on
the quartic saddle, from the same starting point. The three axes are independent
in the package, so the grid is their product and nothing is left out because it
was expected to fail.

A configuration that raises is an outcome. It is reported with its rule, its
model and its subsolver named, and it is not dropped. Nothing here filters on
success.

The limit comes from `nearest_crit`, so it is a name and a distance rather than a
verdict. A run that exhausted its budget while still moving is not a run that
converged somewhere else, and the distance column is what says which happened.

In [1]:
const SADDLEDIR = @__DIR__
using Pkg; Pkg.activate(joinpath(SADDLEDIR, "..", ".."))
include(joinpath(SADDLEDIR, "..", "saddle_problem.jl"))

const CM_RULES = [
    ("RDelta",              () -> RDelta(γ1 = G1, γ2 = G2, γ3 = G3, Δmin = 0.0)),
    ("RStep",               () -> RStep(γ1 = G1, γ2 = G2, γ3 = G3, Δmin = 0.0)),
    ("RDeltaStep",          () -> RDeltaStep(γ1 = G1, γ2 = G2, γ3 = G3, Δmin = 0.0)),
    ("RDFO",                () -> RDFO(γ1 = G1, γ2 = G2, γ3 = G3, ζ = ZETA, Δmin = 0.0)),
    ("RGrad",               () -> RGrad(γ1 = G1, γ2 = G2, γ3 = G3, μ = MU0, Δmin = 0.0)),
    ("RGradCapped",         () -> RGradCapped(γ1 = G1, γ2 = G2, γ3 = G3, μ = MU0,
                                              μ_max = MU_BAR, Δmin = 0.0)),
    ("RAdaptiveStep",       () -> RAdaptiveStep(λ1 = 5.0, λ2 = 5.0, Δmin = 0.0)),
    ("RAdaptiveGrad",       () -> RAdaptiveGrad(μ = MU0, λ1 = 5.0, λ2 = 5.0, Δmin = 0.0)),
    ("RAdaptiveGradCapped", () -> RAdaptiveGradCapped(μ = MU0, μ_max = MU_BAR,
                                                      λ1 = 5.0, λ2 = 5.0, Δmin = 0.0)),
    ("RRTR",                () -> RRTR(γ1 = G1, γ2 = G2, γ3 = G3, Δmin = 0.0)),
    ("RRTRGrad",            () -> RRTRGrad(γ1 = G1, γ2 = G2, γ3 = G3, μ = MU0, Δmin = 0.0)),
]

const CM_MODELS = [
    ("ExactHessian",   () -> ExactHessian()),
    ("SR1",            () -> SR1Model(mem = 10)),
    ("LBFGS",          () -> LBFGSModel(mem = 5)),
    # ("ScaledIdentity", () -> ScaledIdentity(c = 1.0)),
    ("SPDTarget",      () -> SPDTarget(target = SADDLE)),
]

const CM_SUBS = [
    ("SteihaugCG", () -> SteihaugCG()),
    ("ExactMS",    () -> ExactMS()),
    # ("KrylovCR",   () -> KrylovCR()),
    ("EigenPoint", () -> EigenPoint(SteihaugCG())),
]

@printf("%d rules x %d models x %d subsolvers = %d configurations\n",
        length(CM_RULES), length(CM_MODELS), length(CM_SUBS),
        length(CM_RULES) * length(CM_MODELS) * length(CM_SUBS))

  Activating project at `c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\notebooks`


11 rules x 4 models x 3 subsolvers = 132 configurations


## 1. The grid

`tr_solve` runs every cell. The exception handler catches what the solver lets
through and records the exception type as the status, so a configuration that
cannot be built appears in the table with a reason.

In [2]:
"""
    cm_run(rname, mkr, mnm, mkm, snm, mks) -> NamedTuple

One configuration. Errors become rows, with the exception type as the status.
"""
function cm_run(rname, mkr, mnm, mkm, snm, mks)
    t0 = time()
    try
        st = run_cfg(rule = mkr(), model = mkm(), subsolver = mks())
        nc = nearest_crit(st.solution)
        ss = st.solver_specific
        return (rule = rname, model = mnm, sub = snm, raised = false,
                status = string(st.status), msg = "", iter = st.iter,
                limit = nc.name, dist = nc.dist,
                gnorm = Float64(st.dual_feas), obj = Float64(st.objective),
                tail = active_fraction(st; tail = 0.1),
                delta_end = ss[:delta_trajectory][end],
                secs = time() - t0)
    catch err
        err isa InterruptException && rethrow()
        buf = IOBuffer(); showerror(buf, err)
        msg = replace(first(String(take!(buf)), 110), "\n" => " ")
        return (rule = rname, model = mnm, sub = snm, raised = true,
                status = string(nameof(typeof(err))), msg = msg, iter = -1,
                limit = "n/a", dist = NaN, gnorm = NaN, obj = NaN, tail = NaN,
                delta_end = NaN, secs = time() - t0)
    end
end

@info "running the configuration matrix"
const CM_T0 = time()
CM = NamedTuple[]
for (snm, mks) in CM_SUBS, (mnm, mkm) in CM_MODELS, (rname, mkr) in CM_RULES
    push!(CM, cm_run(rname, mkr, mnm, mkm, snm, mks))
end
@info "matrix done" seconds = round(time() - CM_T0, digits = 1) rows = length(CM)

┌ Info: running the configuration matrix
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\notebooks\FINAL\Conf_matrix\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W3sZmlsZQ==.jl:30
┌ Warning: τ ≡ ‖g‖: this model is positive (semi)definite by construction, so λ_min(B) ≥ 0 always and the second-order machinery is a no-op. A :second_order status from this run certifies nothing about ∇²f. Use ExactHessian or SR1Model when the second-order question is the question.
│   model = LBFGSModel
└ @ TrustRegionRadius c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\src\Trust-region\common.jl:316
┌ Info: matrix done
│   seconds = 174.1
│   rows = 132
└ @ Main c:\Users\jerem\OneDrive\Desktop\GitHub\TrustRegionRadius.jl\notebooks\FINAL\Conf_matrix\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W3sZmlsZQ==.jl:36


## 2. The table

`f(x*)` is the objective at the returned point. The three known values are
$f(0,0) = 0$, $f(0.75, 0) = -27/256 = -0.10546875$ and $-1/8$ at either
minimiser.

`dist` and `|g|` stand beside the limit, so `other` is never read as an outcome
on its own.

In [3]:
function cm_table(rows; title = "")
    isempty(title) || (println("\n", "="^126); println(title); println("="^126))
    @printf("%-22s %-15s %-11s %13s %6s | %-8s %10s %10s | %13s %9s\n",
            "rule", "model", "subsolver", "status", "iter",
            "limit", "dist", "|g|", "f(x*)", "tail act")
    println("-"^126)
    for r in rows
        if r.raised
            @printf("%-22s %-15s %-11s %13s %6s | %-8s %10s %10s | %13s %9s\n",
                    r.rule, r.model, r.sub, r.status, "n/a", "n/a", "n/a", "n/a",
                    "n/a", "n/a")
        else
            @printf("%-22s %-15s %-11s %13s %6d | %-8s %10.2e %10.2e | %13.8f %9.4f\n",
                    r.rule, r.model, r.sub, r.status, r.iter, r.limit, r.dist,
                    r.gnorm, r.obj, r.tail)
        end
    end
end

for (snm, _) in CM_SUBS
    cm_table([r for r in CM if r.sub == snm];
             title = "Subsolver: $snm")
end


Subsolver: SteihaugCG
rule                   model           subsolver          status   iter | limit          dist        |g| |         f(x*)  tail act
------------------------------------------------------------------------------------------------------------------------------
RDelta                 ExactHessian    SteihaugCG    first_order     16 | origin     1.30e-05   5.06e-10 |    0.00000000    0.0000
RStep                  ExactHessian    SteihaugCG    first_order     16 | origin     1.30e-05   5.06e-10 |    0.00000000    0.0000
RDeltaStep             ExactHessian    SteihaugCG    first_order     16 | origin     1.30e-05   5.06e-10 |    0.00000000    0.0000
RDFO                   ExactHessian    SteihaugCG    first_order     16 | origin     1.30e-05   5.06e-10 |    0.00000000    0.0000
RGrad                  ExactHessian    SteihaugCG    first_order     16 | origin     1.30e-05   5.03e-10 |    0.00000000    0.0000
RGradCapped            ExactHessian    SteihaugCG       max_iter

## 3. What raised

Every configuration that raised, with the three axes named, the exception type
and its message. These are outcomes of the grid and belong in the count of what
was run.

Two of the three axes can be incompatible with each other, which the grid is
the only way to find out. `ScaledIdentity` returns a `UniformScaling` from
`hessian_op`, and the Krylov solvers ask their operator for its `size`, which a
`UniformScaling` does not have. That pair cannot be run, and the package does
not say so anywhere before the run. `SPDTarget` raises where the target stops
lying downhill, which is a property of the trajectory rather than of the pair,
so it strikes one rule and not the others.

In [4]:
let bad = [r for r in CM if r.raised]
    println("\n", "="^96)
    @printf("%d of %d configurations raised.\n", length(bad), length(CM))
    println("="^96)
    if isempty(bad)
        println("None.")
    else
        @printf("%-22s %-15s %-13s %-13s %s\n",
                "rule", "model", "subsolver", "exception", "message")
        println("-"^150)
        for r in bad
            @printf("%-22s %-15s %-13s %-13s %s\n",
                    r.rule, r.model, r.sub, r.status, r.msg)
        end
        println()
        for (nm, key) in (("by model", :model), ("by subsolver", :sub), ("by rule", :rule))
            d = Dict{String, Int}()
            for r in bad
                k = getproperty(r, key); d[k] = get(d, k, 0) + 1
            end
            @printf("%-14s %s\n", nm, sort(collect(d), by = last, rev = true))
        end
    end
end


3 of 132 configurations raised.
rule                   model           subsolver     exception     message
------------------------------------------------------------------------------------------------------------------------------------------------------
RRTRGrad               SPDTarget       SteihaugCG    DomainError   DomainError with -0.021212385846259098: no SPD model has its minimiser at the target here: φ(x) ≥ 0
RRTRGrad               SPDTarget       ExactMS       DomainError   DomainError with -0.00012222347968789056: no SPD model has its minimiser at the target here: φ(x) ≥ 0
RRTRGrad               SPDTarget       EigenPoint    DomainError   DomainError with -0.021212385846259098: no SPD model has its minimiser at the target here: φ(x) ≥ 0

by model       ["SPDTarget" => 3]
by subsolver   ["SteihaugCG" => 1, "ExactMS" => 1, "EigenPoint" => 1]
by rule        ["RRTRGrad" => 3]


## 4. Outcomes, counted

The composition of the grid, with no filtering. `status` is the solver's own and
`limit` is the classifier's, and the two answer different questions. A run can
report `first_order` at a saddle, which is correct for the criterion it was
given.

In [5]:
let
    println("\n", "="^70)
    println("Status composition over all $(length(CM)) configurations")
    println("="^70)
    d = Dict{String, Int}()
    for r in CM; d[r.status] = get(d, r.status, 0) + 1; end
    for (k, v) in sort(collect(d), by = last, rev = true)
        @printf("  %-16s %4d   %6.1f%%\n", k, v, 100v / length(CM))
    end

    println("\n", "="^70)
    ok = [r for r in CM if !r.raised]
    println("Limit composition, over the $(length(ok)) runs that ran")
    println("="^70)
    e = Dict{String, Int}()
    for r in ok; e[r.limit] = get(e, r.limit, 0) + 1; end
    for (k, v) in sort(collect(e), by = last, rev = true)
        @printf("  %-16s %4d   %6.1f%%\n", k, v, 100v / max(length(ok), 1))
    end

    println("\nThe `other` rows in full, since `other` is not an outcome on its own:")
    oth = [r for r in ok if r.limit == "other"]
    if isempty(oth)
        println("  none")
    else
        @printf("  %-22s %-15s %-11s %13s %6s %10s %10s\n",
                "rule", "model", "subsolver", "status", "iter", "dist", "|g|")
        for r in oth
            @printf("  %-22s %-15s %-11s %13s %6d %10.2e %10.2e\n",
                    r.rule, r.model, r.sub, r.status, r.iter, r.dist, r.gnorm)
        end
    end
end


Status composition over all 132 configurations
  first_order       108     81.8%
  stalled            15     11.4%
  max_iter            6      4.5%
  DomainError         3      2.3%

Limit composition, over the 129 runs that ran
  min+               66     51.2%
  origin             33     25.6%
  saddle             30     23.3%

The `other` rows in full, since `other` is not an outcome on its own:
  none


## 5. Which axis decides the limit

Three cross-tabulations of the limit point, one per axis. If the model decides
and the radius rule does not, the model table has structure and the rule table
does not.

In [6]:
function cross_tab(rows, key, keyname, order)
    limits = sort(unique(r.limit for r in rows))
    println("\n", "="^88)
    println("Limit point by $keyname")
    println("="^88)
    @printf("%-22s", keyname)
    for l in limits; @printf("%10s", l); end
    @printf("%10s\n", "raised")
    println("-"^88)
    for k in order
        sel = [r for r in rows if getproperty(r, key) == k]
        @printf("%-22s", k)
        for l in limits
            @printf("%10d", count(r -> !r.raised && r.limit == l, sel))
        end
        @printf("%10d\n", count(r -> r.raised, sel))
    end
end

cross_tab(CM, :rule,  "rule",      [n for (n, _) in CM_RULES])
cross_tab(CM, :model, "model",     [n for (n, _) in CM_MODELS])
cross_tab(CM, :sub,   "subsolver", [n for (n, _) in CM_SUBS])


Limit point by rule
rule                        min+       n/a    origin    saddle    raised
----------------------------------------------------------------------------------------
RDelta                         6         0         3         3         0
RStep                          6         0         3         3         0
RDeltaStep                     6         0         3         3         0
RDFO                           6         0         3         3         0
RGrad                          6         0         3         3         0
RGradCapped                    6         0         3         3         0
RAdaptiveStep                  6         0         3         3         0
RAdaptiveGrad                  6         0         3         3         0
RAdaptiveGradCapped            6         0         3         3         0
RRTR                           6         0         3         3         0
RRTRGrad                       6         0         3         0         3

Limit point b

In [7]:
println("\nDONE.")


DONE.
